In [ ]:
import os
import cv2
import numpy as np

# SAM2 imports
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


def ensure_dir(p: str) -> None:
    os.makedirs(p, exist_ok=True)


def main():
    image_path = "images/handbag.jpg"
    out_dir = "outputs/sam2"
    masks_dir = os.path.join(out_dir, "masks")
    overlays_dir = os.path.join(out_dir, "overlays")
    ensure_dir(masks_dir)
    ensure_dir(overlays_dir)

    # Load image
    bgr = cv2.imread(image_path)
    if bgr is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Load SAM2
    # config file 和 checkpoint 依你下載的版本調整
    model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
    checkpoint = "checkpoints/sam2.1_hiera_large.pt"

    sam2 = build_sam2(model_cfg, checkpoint, device="cuda")  # 沒 GPU 改成 "cpu"

    mask_generator = SAM2AutomaticMaskGenerator(
        sam2,
        points_per_side=32,
        pred_iou_thresh=0.88,
        stability_score_thresh=0.95,
        crop_n_layers=1,
        crop_n_points_downscale_factor=2,
        min_mask_region_area=100,
    )

    # Generate masks
    masks = mask_generator.generate(image)

    # Save masks
    for i, mask in enumerate(masks, start=1):
        m = (mask["segmentation"].astype(np.uint8) * 255)
        cv2.imwrite(os.path.join(masks_dir, f"handbag_01_mask_{i:03d}.png"), m)

    # Overlay (simple random color overlay)
    overlay = image.copy()
    rng = np.random.default_rng(0)  # 固定隨機種子，方便重現
    for mask in masks:
        color = rng.integers(0, 255, size=3, dtype=np.uint8)
        overlay[mask["segmentation"]] = color

    overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(overlays_dir, "handbag_01_overlay.jpg"), overlay_bgr)

    print(f"Saved {len(masks)} masks to: {masks_dir}")
    print(f"Saved overlay to: {os.path.join(overlays_dir, 'handbag_01_overlay.jpg')}")
if __name__ == "__main__":
    main()

C:\Users\Foxconn\Downloads\sam2-main\sam2-main\sam2\sam2_image_predictor.py:431: UserWarning: cannot import name '_C' from 'sam2' (C:\Users\Foxconn\Downloads\sam2-main\sam2-main\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  masks = self._transforms.postprocess_masks(
